# Check for time_to_swot


In [2]:
import numpy as np
import pandas as pd
import xarray as xr

import matplotlib.pyplot as plt

import os
from glob import glob

from cstes import swot_dir, drifters_dir, get_proj, lonlat2xy, zarr_dir, surface_drifters, depth_drifters, depth_100, depth_50, images_dir, U2, err_acc
from swot import browse_swot_250, add_mask_inside_swot, build_swath_polygon
from diagnosis import dataset_coloc_combs, DRIFTER, ALTI, WD, compute_mean_square, drifters_sources, assign_attrs, compute_mean_square_groupby, dtypes

import cartopy.crs as ccrs
import cartopy.feature as cfeature
import cartopy.geodesic as cgeo
crs = ccrs.PlateCarree()

import cartopy.geodesic as geod
import cartopy.crs as ccrs
import cartopy.feature as cfeature

import dask.dataframe as dd

import pyproj
from pyproj import Geod

from rasterio.transform import Affine

import pynsitu as pyn

from diagnosis import synthetic_figure
from cstes import U2, c0

________
# CHOOSE PARAMETERS HERE

In [2]:
# TO CHOOSE 
drifters_sources = 'all_med_variational_10min_v0.nc'
spectral_decomp = True
#dt = '12h' #'nearestswath'
dt = '10d' #'nearestswath'

_______
# Data 

In [3]:
dfr = dd.read_csv(DRIFTER['nofilter'], parse_dates=['datetime', 'cycle_date'], dtype=dtypes).set_index('row_number')[['cycle_date', 'cycle_number', 'datetime', 'drifter_id', 'time_to_swot']]

In [67]:
dfr = dfr.compute()

In [36]:
dfr_ = dfr.compute().iloc[:100]

In [37]:
dfr_

,cycle_date,cycle_number,datetime,drifter_id,time_to_swot
row_number,,,,,
0,2023-04-01 23:42:32.696536,478,2023-04-10 20:00:00,300534060113380,8 days 20:17:27.303464
1,2023-04-01 23:42:32.696536,478,2023-04-10 20:10:00,300534060113380,8 days 20:27:27.303464
2,2023-04-01 23:42:32.696536,478,2023-04-10 20:20:00,300534060113380,8 days 20:37:27.303464
3,2023-04-01 23:42:32.696536,478,2023-04-10 20:30:00,300534060113380,8 days 20:47:27.303464
4,2023-04-01 23:42:32.696536,478,2023-04-10 20:40:00,300534060113380,8 days 20:57:27.303464
...,...,...,...,...,...
95,2023-04-01 23:42:32.696536,478,2023-04-05 15:40:00,0-4388608,3 days 15:57:27.303464
96,2023-04-01 23:42:32.696536,478,2023-04-05 15:50:00,0-4388608,3 days 16:07:27.303464
97,2023-04-01 23:42:32.696536,478,2023-04-05 16:00:00,0-4388608,3 days 16:17:27.303464


In [24]:
tmin = dfr.groupby('drifter_id').datetime.min().rename('drifter_tmin').compute()
tmax = dfr.groupby('drifter_id').datetime.max().rename('drifter_tmax').compute()

In [90]:
dfr

,cycle_date,cycle_number,datetime,drifter_id,time_to_swot
row_number,,,,,
0,2023-04-01 23:42:32.696536000,478,2023-04-10 20:00:00,300534060113380,8 days 20:17:27.303464
1,2023-04-01 23:42:32.696536000,478,2023-04-10 20:10:00,300534060113380,8 days 20:27:27.303464
2,2023-04-01 23:42:32.696536000,478,2023-04-10 20:20:00,300534060113380,8 days 20:37:27.303464
3,2023-04-01 23:42:32.696536000,478,2023-04-10 20:30:00,300534060113380,8 days 20:47:27.303464
4,2023-04-01 23:42:32.696536000,478,2023-04-10 20:40:00,300534060113380,8 days 20:57:27.303464
...,...,...,...,...,...
7462245,2023-07-09 18:56:21.868455616,577,2023-07-06 08:30:00,4694474,3 days 10:26:21.868455616
7462246,2023-07-09 18:56:21.868455616,577,2023-07-06 08:40:00,4694474,3 days 10:16:21.868455616
7462247,2023-07-09 18:56:21.868455616,577,2023-07-06 08:50:00,4694474,3 days 10:06:21.868455616


In [59]:
ld = (tmax-tmin)

In [61]:
ld[ld>pd.Timedelta('20d')]

drifter_id
0-4351945   22 days 00:30:00
0-4367706   22 days 16:20:00
0-4367707   92 days 20:20:00
0-4388605   27 days 09:00:00
0-4388635   30 days 01:40:00
                  ...       
6204604     29 days 16:50:00
6204605     71 days 11:20:00
6204606     29 days 16:50:00
6204607     51 days 04:20:00
6204608     83 days 22:50:00
Length: 74, dtype: timedelta64[ns]

In [44]:
# Build test table 
cycle_dates = dfr.cycle_date.unique().compute()
drifter_id = dfr.drifter_id.unique().compute()

test_tmin = xr.DataArray(np.full((len(drifter_id),len(cycle_dates)), True), coords={'drifter_id':drifter_id, 'cycle_date':cycle_dates}).rename('drifter_tmin')
test_tmax = xr.DataArray(np.full((len(drifter_id),len(cycle_dates)), True), coords={'drifter_id':drifter_id, 'cycle_date':cycle_dates}).rename('drifter_tmax')

for d in drifter_id :
    for c in cycle_dates:
        test_tmin.loc[d, c] = (c-pd.Timedelta('10d')> tmin.loc[d])
        test_tmax.loc[d, c] = (c+pd.Timedelta('10d')< tmax.loc[d])

In [48]:
dft = pd.concat([test_tmin.to_dataframe(), test_tmax.to_dataframe()], axis=1)

In [53]:
dft['test'] = dft.drifter_tmin & dft.drifter_tmax

In [85]:
dft.test

drifter_id  cycle_date                   
4450426     2023-04-30 19:10:52.420670656    False
            2023-06-16 11:50:28.683088192    False
            2023-06-25 21:07:31.174987456    False
            2023-04-24 20:07:05.352417216    False
            2023-07-02 20:01:56.405524096    False
                                             ...  
4694458     2023-07-06 19:24:28.273369600    False
            2023-06-15 11:59:51.056402560    False
            2023-05-24 15:25:59.365211712    False
            2023-05-27 01:48:35.485000320    False
            2023-05-30 01:20:28.891069568    False
Name: test, Length: 27507, dtype: bool

In [86]:
dfr.reset_index().set_index(['drifter_id', 'cycle_date'])

row_number  cycle_number  \
drifter_id      cycle_date                                                
300534060113380 2023-04-01 23:42:32.696536000           0           478   
                2023-04-01 23:42:32.696536000           1           478   
                2023-04-01 23:42:32.696536000           2           478   
                2023-04-01 23:42:32.696536000           3           478   
                2023-04-01 23:42:32.696536000           4           478   
...                                                   ...           ...   
4694474         2023-07-09 18:56:21.868455616     7462245           577   
                2023-07-09 18:56:21.868455616     7462246           577   
                2023-07-09 18:56:21.868455616     7462247           577   
                2023-07-09 18:56:21.868455616     7462248           577   
                2023-07-09 18:56:21.868455616     7462249           577   

                                                         datetime  \
drifter_id      cycle_date                                          
300534060113380 2023-04-01 23:42:32.696536000 2023-04-10 20:00:00   
                2023-04-01 23:42:32.696536000 2023-04-10 20:10:00   
                2023-04-01 23:42:32.696536000 2023-04-10 20:20:00   
                2023-04-01 23:42:32.696536000 2023-04-10 20:30:00   
                2023-04-01 23:42:32.696536000 2023-04-10 20:40:00   
...                                                           ...   
4694474         2023-07-09 18:56:21.868455616 2023-07-06 08:30:00   
                2023-07-09 18:56:21.868455616 2023-07-06 08:40:00   
                2023-07-09 18:56:21.868455616 2023-07-06 08:50:00   
                2023-07-09 18:56:21.868455616 2023-07-06 09:00:00   
                2023-07-09 18:56:21.868455616 2023-07-06 09:10:00   

                                                            time_to_swot  
drifter_id      cycle_date                                                
300534060113380 2023-04-01 23:42:32.696536000     8 days 20:17:27.303464  
                2023-04-01 23:42:32.696536000     8 days 20:27:27.303464  
                2023-04-01 23:42:32.696536000     8 days 20:37:27.303464  
                2023-04-01 23:42:32.696536000     8 days 20:47:27.303464  
                2023-04-01 23:42:32.696536000     8 days 20:57:27.303464  
...                                                                  ...  
4694474         2023-07-09 18:56:21.868455616  3 days 10:26:21.868455616  
                2023-07-09 18:56:21.868455616  3 days 10:16:21.868455616  
                2023-07-09 18:56:21.868455616  3 days 10:06:21.868455616  
                2023-07-09 18:56:21.868455616  3 days 09:56:21.868455616  
                2023-07-09 18:56:21.868455616  3 days 09:46:21.868455616  

[7462250 rows x 4 columns]

In [69]:
dfr.reset_index().set_index(['drifter_id', 'cycle_date']).where(dft.test).dropna()

row_number  cycle_number  \
drifter_id      cycle_date                                                
300534060315840 2023-04-10 22:18:14.486964224    349930.0         487.0   
                2023-04-10 22:18:14.486964224    349931.0         487.0   
                2023-04-10 22:18:14.486964224    349932.0         487.0   
                2023-04-10 22:18:14.486964224    349933.0         487.0   
                2023-04-10 22:18:14.486964224    349934.0         487.0   
...                                                   ...           ...   
300534060015760 2023-07-08 19:05:44.003893184   7442943.0         576.0   
                2023-07-08 19:05:44.003893184   7442944.0         576.0   
                2023-07-08 19:05:44.003893184   7442945.0         576.0   
                2023-07-08 19:05:44.003893184   7442946.0         576.0   
                2023-07-08 19:05:44.003893184   7442947.0         576.0   

                                                         datetime  \
drifter_id      cycle_date                                          
300534060315840 2023-04-10 22:18:14.486964224 2023-03-31 22:20:00   
                2023-04-10 22:18:14.486964224 2023-03-31 22:30:00   
                2023-04-10 22:18:14.486964224 2023-03-31 22:40:00   
                2023-04-10 22:18:14.486964224 2023-03-31 22:50:00   
                2023-04-10 22:18:14.486964224 2023-03-31 23:00:00   
...                                                           ...   
300534060015760 2023-07-08 19:05:44.003893184 2023-07-18 18:20:00   
                2023-07-08 19:05:44.003893184 2023-07-18 18:30:00   
                2023-07-08 19:05:44.003893184 2023-07-18 18:40:00   
                2023-07-08 19:05:44.003893184 2023-07-18 18:50:00   
                2023-07-08 19:05:44.003893184 2023-07-18 19:00:00   

                                                            time_to_swot  
drifter_id      cycle_date                                                
300534060315840 2023-04-10 22:18:14.486964224  9 days 23:58:14.486964224  
                2023-04-10 22:18:14.486964224  9 days 23:48:14.486964224  
                2023-04-10 22:18:14.486964224  9 days 23:38:14.486964224  
                2023-04-10 22:18:14.486964224  9 days 23:28:14.486964224  
                2023-04-10 22:18:14.486964224  9 days 23:18:14.486964224  
...                                                                  ...  
300534060015760 2023-07-08 19:05:44.003893184  9 days 23:14:15.996106816  
                2023-07-08 19:05:44.003893184  9 days 23:24:15.996106816  
                2023-07-08 19:05:44.003893184  9 days 23:34:15.996106816  
                2023-07-08 19:05:44.003893184  9 days 23:44:15.996106816  
                2023-07-08 19:05:44.003893184  9 days 23:54:15.996106816  

[3292104 rows x 4 columns]

In [88]:
def remove_dt_traj_limit_coloc(dfr, dt):
    """ 
    Remove colocations that are at the dt-time limit of a drifter trajectories, for all cycle
    Example dfr  :
    dfr = dd.read_csv(DRIFTER['nofilter'], parse_dates=['datetime', 'cycle_date'], dtype=dtypes).set_index('row_number')[['cycle_date','datetime', 'drifter_id']]
    """
    # Build test table 
    cycle_dates = dfr.cycle_date.unique()
    drifter_id = dfr.drifter_id.unique()

    test_tmin = xr.DataArray(np.full((len(drifter_id),len(cycle_dates)), True), coords={'drifter_id':drifter_id, 'cycle_date':cycle_dates}).rename('drifter_tmin')
    test_tmax = xr.DataArray(np.full((len(drifter_id),len(cycle_dates)), True), coords={'drifter_id':drifter_id, 'cycle_date':cycle_dates}).rename('drifter_tmax')

    for d in drifter_id :
        for c in cycle_dates:
            test_tmin.loc[d, c] = (c-pd.Timedelta('10d')> tmin.loc[d])
            test_tmax.loc[d, c] = (c+pd.Timedelta('10d')< tmax.loc[d])
    dft = pd.concat([test_tmin.to_dataframe(), test_tmax.to_dataframe()], axis=1)
    dft['test'] = dft.drifter_tmin & dft.drifter_tmax

    return dfr.reset_index().set_index(['drifter_id', 'cycle_date']).where(dft.test).dropna().reset_index().set_index('row_number')

In [89]:
remove_dt_traj_limit_coloc(dfr, pd.Timedelta('10d'))

,drifter_id,cycle_date,cycle_number,datetime,time_to_swot
row_number,,,,,
349930.0,300534060315840,2023-04-10 22:18:14.486964224,487.0,2023-03-31 22:20:00,9 days 23:58:14.486964224
349931.0,300534060315840,2023-04-10 22:18:14.486964224,487.0,2023-03-31 22:30:00,9 days 23:48:14.486964224
349932.0,300534060315840,2023-04-10 22:18:14.486964224,487.0,2023-03-31 22:40:00,9 days 23:38:14.486964224
349933.0,300534060315840,2023-04-10 22:18:14.486964224,487.0,2023-03-31 22:50:00,9 days 23:28:14.486964224
349934.0,300534060315840,2023-04-10 22:18:14.486964224,487.0,2023-03-31 23:00:00,9 days 23:18:14.486964224
...,...,...,...,...,...
7442943.0,300534060015760,2023-07-08 19:05:44.003893184,576.0,2023-07-18 18:20:00,9 days 23:14:15.996106816
7442944.0,300534060015760,2023-07-08 19:05:44.003893184,576.0,2023-07-18 18:30:00,9 days 23:24:15.996106816
7442945.0,300534060015760,2023-07-08 19:05:44.003893184,576.0,2023-07-18 18:40:00,9 days 23:34:15.996106816
